# Verifikasi P1, P2, dan P3

Jalankan dari atas ke bawah. Data bersama dimuat dari salinan lokal atau repositori publik. Untuk tolok ukur lokal, versi NumPy, SciPy, dan sistem dicetak bersama hasil.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from csv import DictWriter
from hashlib import sha256
from itertools import combinations, product
from pathlib import Path
from random import Random
from time import perf_counter
from urllib.request import urlopen
import json
import platform

import numpy as np
import scipy
from scipy.optimize import Bounds, LinearConstraint, linprog, milp

PUBLIC_DATA_URL = (
    "https://raw.githubusercontent.com/kiittkiidd/Lampiran-Model-Matematika-Tesis/"
    "main/case/Data_Bersama.json"
)
DATA_SHA256 = "860d18438c471197f4cda3a3602d945c13acd7d986a06a9bfe1d5275c6dad9e1"
for directory in (Path.cwd(), *Path.cwd().parents):
    data_file = directory / "case/Data_Bersama.json"
    if data_file.is_file():
        data_bytes = data_file.read_bytes()
        if sha256(data_bytes).hexdigest() == DATA_SHA256:
            break
else:
    with urlopen(PUBLIC_DATA_URL, timeout=30) as response:
        data_bytes = response.read()
if sha256(data_bytes).hexdigest() != DATA_SHA256:
    raise ValueError("Data bersama tidak cocok dengan versi notebook ini.")
DATA = json.loads(data_bytes)
PRODUCTS = DATA["produk"]
WAREHOUSES = range(1, 5)
CUSTOMERS = range(1, 7)
HIGH_IN = DATA["level"]["H"]["jalur_masuk"]
BALANCES = sorted({(g, p) for _, g, p in HIGH_IN})
DEMAND_KEYS = list(product(CUSTOMERS, PRODUCTS))
SCENARIOS = DATA["P2"]["skenario"]
TOL = 1e-7

## 1. Arus, biaya, dan skenario

Bangun R1–R39 berdasarkan rute aktif. P2 memakai empat keputusan buka yang sama di tiga skenario. P3 memakai jaringan dan permintaan pasti P1.

In [ ]:
def route_matrices(inbound, outbound):
    """Bentuk 39 posisi baris pada urutan R1–R39 yang dipakai P1."""
    flows = [("x", *a) for a in inbound] + [("z", *a) for a in outbound]
    a = np.zeros((39, len(flows)))
    warehouse_coefficient = np.zeros((39, 4))
    for col, (kind, i, j, p) in enumerate(flows):
        if kind == "x":
            a[i - 1, col] = 1
            a[6 + BALANCES.index((j, p)), col] = 1
        else:
            a[6 + BALANCES.index((i, p)), col] = -1
            a[17 + DEMAND_KEYS.index((j, p)), col] = 1
            a[34 + i, col] = 1
    warehouse_coefficient[35:, :] = -np.diag(DATA["kapasitas_gudang"])
    return a, warehouse_coefficient


def rhs(demand):
    values = [demand.get(f"{j},{p}", 0.0) for j, p in DEMAND_KEYS]
    return np.array(DATA["kapasitas_pabrik"] + [0] * 11 + values + [0] * 4, dtype=float)


def operating_costs(inbound, outbound):
    return np.array(
        [DATA["tarif_masuk_per_palet"][i - 1][g - 1] for i, g, _ in inbound]
        + [
            DATA["tarif_keluar_per_palet"][g - 1][j - 1]
            + DATA["tarif_penanganan_per_palet"][g - 1][PRODUCTS.index(p)]
            for g, j, p in outbound
        ]
    )


def expanded_rows(a, gy, demands):
    """Susun baris skenario dengan satu set y yang dipakai bersama."""
    n = a.shape[1]
    matrix = np.zeros((39 * len(demands), 4 + n * len(demands)))
    row_upper = np.concatenate([rhs(demand) for demand in demands])
    row_lower = row_upper.copy()
    for scenario_index, _ in enumerate(demands):
        start = 39 * scenario_index
        block = slice(start, start + 39)
        flow_cols = slice(4 + scenario_index*n, 4 + (scenario_index+1)*n)
        matrix[block, :4] = gy
        matrix[block, flow_cols] = a
        row_lower[start:start+6] = -np.inf
        row_lower[start+35:start+39] = -np.inf
    return matrix, row_lower, row_upper


def enumerate_warehouses(a, gy, costs, demands, probabilities):
    """Enumerasi 16 pola y; selesaikan satu LP arus untuk setiap skenario."""
    choices = {}
    upper = np.r_[0:6, 35:39]
    for opened in product((0, 1), repeat=4):
        scenario_costs = []
        for demand in demands:
            residual = rhs(demand) - gy @ opened
            result = linprog(
                costs, A_ub=a[upper], b_ub=residual[upper],
                A_eq=a[6:35], b_eq=residual[6:35], bounds=(0, None),
                method="highs",
            )
            if result.status == 2:
                break
            if not result.success:
                raise RuntimeError(result.message)
            scenario_costs.append(result.fun)
        if len(scenario_costs) == len(demands):
            opening = float(np.dot(DATA["biaya_buka_gudang"], opened))
            choices[opened] = (
                opening + float(np.dot(probabilities, scenario_costs)),
                opening,
                scenario_costs,
            )
    return choices

## 2. Penyelesaian dan pembanding

P1 dan P2 memakai satu MILP per tingkat. P3 memakai dua penyelesaian per titik untuk memilih solusi dengan emisi terendah pada biaya optimal, atau memilih biaya terendah pada emisi minimum. Enumerasi terpisah memeriksa optimum dengan empat keputusan lokasi tetap.

In [ ]:
def decisions(inbound, outbound):
    return (
        [f"y[{g}]" for g in WAREHOUSES]
        + [f"x[{i},{g},{p}]" for i, g, p in inbound]
        + [f"z[{g},{j},{p}]" for g, j, p in outbound]
    )


def emissions(inbound, outbound):
    data = DATA["P3"]
    return np.asarray(
        data["emisi_tetap_gudang"]
        + [data["emisi_masuk_per_palet"][i - 1][g - 1] for i, g, _ in inbound]
        + [data["emisi_keluar_per_palet"][g - 1][j - 1] for g, j, _ in outbound]
    )


def problem(a, gy, demands, probabilities, operating):
    matrix, lower, upper = expanded_rows(a, gy, demands)
    costs = np.r_[
        DATA["biaya_buka_gudang"],
        np.concatenate([prob * operating for prob in probabilities]),
    ]
    return matrix, lower, upper, costs


def solve(instance, objective, limits=(), relaxed=False):
    start = perf_counter()
    matrix, lower, upper, _ = instance
    if limits:
        matrix = np.vstack([matrix, *[row for row, _ in limits]])
        upper = np.r_[upper, [bound for _, bound in limits]]
        lower = np.r_[lower, np.full(len(limits), -np.inf)]
    n = len(objective)
    integrality = np.zeros(n)
    if not relaxed:
        integrality[:4] = 1
    constraints = LinearConstraint(matrix, lower, upper)
    bounds = Bounds(0, np.r_[np.ones(4), np.full(n - 4, np.inf)])
    prepared_at = perf_counter()
    result = milp(
        objective, integrality=integrality, constraints=constraints, bounds=bounds,
        options={"mip_rel_gap": 0},
    )
    ended_at = perf_counter()
    if not result.success:
        raise RuntimeError(result.message)
    lhs = matrix @ result.x
    assert np.all(lhs >= lower - 2e-6) and np.all(lhs <= upper + 2e-6)
    assert np.all(result.x >= -2e-6) and np.all(result.x[:4] <= 1 + 2e-6)
    assert np.isclose(result.fun, objective @ result.x, atol=TOL, rtol=0)
    if not relaxed:
        assert np.allclose(result.x[:4], np.round(result.x[:4]), atol=TOL, rtol=0)
    return result, {
        "persiapan_ms": (prepared_at - start) * 1000,
        "solver_ms": (ended_at - prepared_at) * 1000,
        "node": int(getattr(result, "mip_node_count", 0) or 0),
    }


def lexicographic(instance, first, second, limits=()):
    first_result, timing_1 = solve(instance, first, limits)
    second_result, timing_2 = solve(instance, second, (*limits, (first, first_result.fun + TOL)))
    return first_result, second_result, (timing_1, timing_2)


def frontier(instance, emission):
    cost = instance[3]
    low_emission, green, green_timings = lexicographic(instance, emission, cost)
    _, cheap, cheap_timings = lexicographic(instance, cost, emission)
    minimum, maximum = low_emission.fun, float(emission @ cheap.x)
    assert minimum < maximum - TOL
    points = [(minimum, green, green_timings)]
    for fraction in DATA["P3"]["fraksi_batas_emisi"][1:-1]:
        cap = minimum + fraction * (maximum - minimum)
        _, submission, timings = lexicographic(instance, cost, emission, ((emission, cap),))
        points.append((cap, submission, timings))
    return [*points, (maximum, cheap, cheap_timings)]


def min_over_patterns(a, gy, demand, objective, limits=()):
    """Validasi optimum lewat 16 pola gudang dengan keputusan lokasi tetap."""
    upper = np.r_[0:6, 35:39]
    matrix = np.vstack([a[upper], *[row[4:] for row, _ in limits]])
    bounds = np.array([bound for _, bound in limits])
    candidates = []
    for y in product((0, 1), repeat=4):
        residual = rhs(demand) - gy @ y
        extra = bounds - [row[:4] @ y for row, _ in limits]
        lp = linprog(
            objective[4:], A_ub=matrix, b_ub=np.r_[residual[upper], extra],
            A_eq=a[6:35], b_eq=residual[6:35], bounds=(0, None), method="highs",
        )
        if lp.status != 2:
            if not lp.success:
                raise RuntimeError(lp.message)
            candidates.append(float(objective[:4] @ y + lp.fun))
    return min(candidates)


def portfolio(choices):
    best = min(record[0] for record in choices.values())
    opened = lambda y: "+".join(f"G{i}" for i, flag in enumerate(y, start=1) if flag)
    ordered = sorted([(opened(y), record[0]) for y, record in choices.items()], key=lambda v: v[1])
    return {
        "layak": len(choices),
        "wajib_buka": [f"G{i}" for i in WAREHOUSES if all(y[i - 1] for y in choices)],
        "dalam_5_persen": sum(value <= 1.05 * best + TOL for _, value in ordered),
        "kombinasi_dan_biaya": ordered,
    }


def grouped_timings(timings):
    prep = sum(record["persiapan_ms"] for record in timings)
    solve_sum = sum(record["solver_ms"] for record in timings)
    return {
        "persiapan_ms": prep,
        "solver_ms": solve_sum,
        "node": sum(record["node"] for record in timings),
    }


def open_warehouses(solution):
    return [f"G{g}" for g in WAREHOUSES if round(solution.x[g - 1])]


def scenarios(mean):
    key = DATA["P2"]["sasaran_permintaan"]
    sigma = DATA["P2"]["simpangan_baku"]
    demands = [{**mean, key: mean[key] + sigma * s["deviasi_baku"]} for s in SCENARIOS]
    probs = [s["peluang"] for s in SCENARIOS]
    assert abs(np.dot(probs, [d[key] for d in demands]) - mean[key]) <= TOL
    return demands, probs

## 3. Hitung pasangan kendala satu per satu

Simpan setiap pasangan beserta seluruh variabel bersama. Jumlah dari irisan himpunan dicek dengan perkalian matriks insidensi.

In [ ]:
def coupling(matrix):
    incidence = (matrix != 0).astype(np.int64)
    weights = incidence @ incidence.T
    upper = np.triu(weights, 1)
    return {
        "bobot": int(upper.sum()),
        "pasangan_positif": int(np.count_nonzero(upper)),
        "baris_aktif": int(np.count_nonzero(incidence.any(axis=1))),
        "baris_acuan": int(incidence.shape[0]),
        "variabel": int(incidence.shape[1]),
        "nnz": int(incidence.sum()),
    }


ROW_LABELS = (
    [f"Kapasitas P{i}" for i in range(1, 7)]
    + [f"Neraca G{g}–{p}" for g, p in BALANCES]
    + [f"Permintaan K{j}–{p}" for j, p in DEMAND_KEYS]
    + [f"Kapasitas G{g}" for g in WAREHOUSES]
)


def row_supports(inbound, outbound, profile):
    a, gy = route_matrices(inbound, outbound)
    names = decisions(inbound, outbound)
    if profile == "P2":
        names = names[:4] + [f"{x}@{s['nama']}" for s in SCENARIOS for x in names[4:]]
        matrix, _, _ = expanded_rows(a, gy, [{}] * len(SCENARIOS))
        row_ids = [f"{s['nama']}:R{i}" for s in SCENARIOS for i in range(1, 40)]
        labels = [label for _ in SCENARIOS for label in ROW_LABELS]
    else:
        matrix = np.c_[gy, a]
        row_ids = [f"R{i}" for i in range(1, 40)]
        labels = ROW_LABELS.copy()
        if profile == "P3":
            matrix = np.vstack([matrix, emissions(inbound, outbound)])
            row_ids.append("R40")
            labels.append("Batas emisi")
    supports = [{v for v, coefficient in zip(names, row) if coefficient != 0} for row in matrix]
    return supports, row_ids, labels, matrix


def export_pairs(profile, output_path):
    reference_outbound = [(g, j, p) for g, p in BALANCES for j in CUSTOMERS]
    allowed, row_ids, labels, ref_matrix = row_supports(HIGH_IN, reference_outbound, profile)
    by_level = {
        key: row_supports(level["jalur_masuk"], level["jalur_keluar"], profile)
        for key, level in DATA["level"].items()
    }
    counts = dict.fromkeys(("acuan", *by_level), 0)
    header = ["baris_r", "arti_r", "baris_s", "arti_s"]
    for key in counts:
        header += [f"bobot_{key}", f"variabel_bersama_{key}"]
    with output_path.open("w", encoding="utf-8", newline="") as output:
        writer = DictWriter(output, fieldnames=header, lineterminator="\n")
        writer.writeheader()
        for r, s in combinations(range(len(row_ids)), 2):
            item = dict(baris_r=row_ids[r], arti_r=labels[r], baris_s=row_ids[s], arti_s=labels[s])
            common = allowed[r] & allowed[s]
            for key in counts:
                subset = common if key == "acuan" else by_level[key][0][r] & by_level[key][0][s]
                assert subset <= common
                counts[key] += len(subset)
                item[f"bobot_{key}"] = len(subset)
                item[f"variabel_bersama_{key}"] = "; ".join(sorted(subset))
            writer.writerow(item)
    assert counts["acuan"] == coupling(ref_matrix)["bobot"]
    for key, (_, _, _, matrix) in by_level.items():
        assert counts[key] == coupling(matrix)["bobot"]
    return counts

## 4. Kelayakan, trade-off, dan benchmark

Cocokkan MILP dengan enumerasi, periksa perluasan pilihan lokasi, lalu ukur 25 pengulangan setelah dua putaran pemanasan. Acak urutan pekerjaan antarprofil pada setiap pengulangan.

In [ ]:
def validate_all():
    return_data, tasks = {}, {}
    previous = {"P1": set(), "P2": set()}
    for level, spec in DATA["level"].items():
        inbound, outbound, mean = (
            spec["jalur_masuk"], spec["jalur_keluar"], spec["permintaan_dasar"]
        )
        a, gy = route_matrices(inbound, outbound)
        operating = operating_costs(inbound, outbound)
        demand_scenarios, probs = scenarios(mean)
        return_data[level] = {}
        for profile, demands, p in (("P1", [mean], [1.0]), ("P2", demand_scenarios, probs)):
            instance = problem(a, gy, demands, p, operating)
            tasks[(profile, level)] = (instance, None)
            result, _ = solve(instance, instance[3])
            relaxed, _ = solve(instance, instance[3], relaxed=True)
            choices = enumerate_warehouses(a, gy, operating, demands, p)
            assert previous[profile] < choices.keys()
            previous[profile] = choices.keys()
            assert abs(result.fun - min(c[0] for c in choices.values())) <= TOL
            record = portfolio(choices)
            record.update(
                biaya=float(result.fun), gudang=open_warehouses(result),
                batas_bawah_lp=float(relaxed.fun),
                gap_lp_persen=100 * (result.fun - relaxed.fun) / abs(result.fun),
                ukuran=coupling(instance[0]),
            )
            return_data[level][profile] = record

        deterministic = tasks[("P1", level)][0]
        emission = emissions(inbound, outbound)
        cost = deterministic[3]
        tasks[("P3", level)] = (deterministic, emission)
        points = frontier(deterministic, emission)
        assert abs(points[-1][1].x @ cost - return_data[level]["P1"]["biaya"]) <= 2e-6
        assert abs(points[0][0] - min_over_patterns(a, gy, mean, emission)) <= 2e-6
        result_points = []
        for cap, solution, _ in points:
            cost_lp = min_over_patterns(a, gy, mean, cost, ((emission, cap + TOL),))
            actual_cost = float(cost @ solution.x)
            actual_emission = float(emission @ solution.x)
            assert abs(cost_lp - actual_cost) <= 2e-6
            lex_emission = min_over_patterns(a, gy, mean, emission, ((cost, actual_cost + TOL),))
            assert abs(lex_emission - actual_emission) <= 2e-6
            result_points.append({
                "epsilon": cap, "biaya": actual_cost, "emisi": actual_emission,
                "gudang": open_warehouses(solution),
            })
        return_data[level]["P3"] = result_points
    return return_data, tasks


def sensitivity_medium():
    """Ubah kapasitas ±10% seragam, dengan rute dan skenario Medium yang sama."""
    spec = DATA["level"]["M"]
    a, base = route_matrices(spec["jalur_masuk"], spec["jalur_keluar"])
    operating = operating_costs(spec["jalur_masuk"], spec["jalur_keluar"])
    demands, probs = scenarios(spec["permintaan_dasar"])
    comparison = []
    for factor in (0.9, 1.0, 1.1):
        gy = base * factor
        instance = problem(a, gy, demands, probs, operating)
        assert np.array_equal(base != 0, gy != 0)
        result, timing = solve(instance, instance[3])
        relaxed, _ = solve(instance, instance[3], relaxed=True)
        choices = enumerate_warehouses(a, gy, operating, demands, probs)
        assert abs(result.fun - min(c[0] for c in choices.values())) <= TOL
        comparison.append({
            "faktor_kapasitas": factor,
            "bobot_WCC_jaringan": coupling(np.c_[gy, a])["bobot"],
            "pola_layak": len(choices),
            "gudang_wajib": [f"G{i}" for i in WAREHOUSES if all(y[i - 1] for y in choices)],
            "node": timing["node"],
            "gap_lp_persen": 100 * (result.fun - relaxed.fun) / abs(result.fun),
        })
    return comparison


def benchmark(tasks, repeats=25):
    rows = []
    rng = Random(42)
    keys = list(tasks)
    for _ in range(2):
        for profile, level in keys:
            instance, emission = tasks[(profile, level)]
            if profile == "P3":
                frontier(instance, emission)
            else:
                solve(instance, instance[3])
    for rep in range(1, repeats + 1):
        rng.shuffle(keys)
        for profile, level in keys:
            instance, emission = tasks[(profile, level)]
            start = perf_counter()
            if profile == "P3":
                timings = [t for _, _, pair in frontier(instance, emission) for t in pair]
            else:
                timings = [solve(instance, instance[3])[1]]
            total = (perf_counter() - start) * 1000
            rows.append({
                "profil": profile, "tingkat": level, "pengulangan": rep,
                "panggilan_MILP": len(timings), **grouped_timings(timings), "total_ms": total,
            })
    return rows


def benchmark_summary(rows):
    by_job = {}
    for key in sorted({(r["profil"], r["tingkat"]) for r in rows}):
        group = [r for r in rows if (r["profil"], r["tingkat"]) == key]
        by_job.setdefault(key[0], {})[key[1]] = {
            "pengulangan": len(group), "panggilan_MILP": group[0]["panggilan_MILP"],
            **{
                field: [float(np.quantile([r[field] for r in group], q)) for q in (0.5, 0.1, 0.9)]
                for field in ("persiapan_ms", "solver_ms", "total_ms", "node")
            },
        }
    return by_job

## 5. Jalankan dan simpan hasil

Waktu solver mengukur panggilan SciPy/HiGHS. Untuk P3, waktu solver adalah jumlah dari 10 panggilan untuk lima titik emisi. Gunakan CSV mentah untuk melihat sebaran pengulangan.

In [ ]:
output = Path("hasil_verifikasi")
output.mkdir(exist_ok=True)
start = perf_counter()
results, tasks = validate_all()
sensitivity = sensitivity_medium()
wcc = {p: export_pairs(p, output / f"Audit_WCC_Pasangan_{p}.csv") for p in ("P1","P2","P3")}
assert wcc == {
    "P1": {"acuan":232, "L":109, "M":142, "H":196},
    "P2": {"acuan":708, "L":339, "M":438, "H":600},
    "P3": {"acuan":502, "L":238, "M":310, "H":430},
}
validation_seconds = perf_counter() - start
timings = benchmark(tasks)
runtime_summary = benchmark_summary(timings)
with (output / "Benchmark_Waktu.csv").open("w", newline="", encoding="utf-8") as file:
    writer = DictWriter(file, fieldnames=list(timings[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(timings)
summary = dict(
    hasil=results, sensitivitas_medium=sensitivity, wcc=wcc,
    benchmark=runtime_summary, validasi_tanpa_benchmark_detik=validation_seconds,
    versi={"Python":platform.python_version(), "NumPy":np.__version__, "SciPy":scipy.__version__,
           "sistem":platform.system(), "arsitektur":platform.machine()},
)
(output / "Ringkasan.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
print("Validasi dan pengukuran selesai.")
print("WCC (bobot acuan, Low, Medium, High)", wcc)
for profile, levels in runtime_summary.items():
    for level, row in levels.items():
        print(f"{profile} {level} | {row['panggilan_MILP']} MILP | {row['solver_ms'][0]:.3f} ms median")